In [1]:
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

crime_full = pd.read_csv("data/cleaned_crime_long_through_june2026.csv")
crime_full["Date"] = pd.to_datetime(crime_full["Date"])

london_full = crime_full.groupby("Date")["CrimeCount"].sum().reset_index().sort_values("Date")

train_2025 = london_full[london_full["Date"] <= "2024-12-31"]
ts_train_2025 = train_2025.set_index("Date")["CrimeCount"]

print("Training data ends at:", ts_train_2025.index.max())
print("Training data length:", len(ts_train_2025), "months")

model_2025 = SARIMAX(ts_train_2025, order=(1,1,0), seasonal_order=(1,1,0,12),
                      enforce_stationarity=False, enforce_invertibility=False)
fit_2025 = model_2025.fit(disp=False)
forecast_2025 = fit_2025.forecast(steps=12)

print("\nSARIMA forecast for 2025:")
print(forecast_2025)

Training data ends at: 2024-12-01 00:00:00
Training data length: 177 months


/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)



SARIMA forecast for 2025:
2025-01-01    77532.310955
2025-02-01    75194.996691
2025-03-01    79458.345248
2025-04-01    76796.544999
2025-05-01    82679.148217
2025-06-01    83438.131544
2025-07-01    84316.854411
2025-08-01    81699.365139
2025-09-01    81106.744212
2025-10-01    82373.668518
2025-11-01    84192.821181
2025-12-01    81158.759557
Freq: MS, Name: predicted_mean, dtype: float64


In [2]:
actual_2025 = london_full[(london_full["Date"] >= "2025-01-01") & (london_full["Date"] <= "2025-12-31")]
actual_2025 = actual_2025.sort_values("Date").reset_index(drop=True)

print(actual_2025)

         Date  CrimeCount
0  2025-01-01       73059
1  2025-02-01       69091
2  2025-03-01       75599
3  2025-04-01       74357
4  2025-05-01       78653
5  2025-06-01       78414
6  2025-07-01       82509
7  2025-08-01       77380
8  2025-09-01       73857
9  2025-10-01       76963
10 2025-11-01       76013
11 2025-12-01       73393


In [3]:
naive_2025 = london_full[(london_full["Date"] >= "2024-01-01") & (london_full["Date"] <= "2024-12-31")]
naive_2025 = naive_2025.sort_values("Date").reset_index(drop=True)

print(naive_2025)

         Date  CrimeCount
0  2024-01-01       73751
1  2024-02-01       73425
2  2024-03-01       75514
3  2024-04-01       74024
4  2024-05-01       78233
5  2024-06-01       77845
6  2024-07-01       80099
7  2024-08-01       78650
8  2024-09-01       76220
9  2024-10-01       81476
10 2024-11-01       80699
11 2024-12-01       76170


In [14]:
comparison_2025 = pd.DataFrame({
    "Month": actual_2025["Date"].dt.strftime("%Y-%m"),
    "Actual": actual_2025["CrimeCount"].values,
    "SARIMA": forecast_2025.round(0).values,
    "Naive_Seasonal": naive_2025["CrimeCount"].values
})

comparison_2025["Ensemble"] = (0.4 * comparison_2025["SARIMA"]) + (0.6 * comparison_2025["Naive_Seasonal"])
comparison_2025["Ensemble"] = comparison_2025["Ensemble"].round(0)

comparison_2025["SARIMA_Error"] = (abs(comparison_2025["Actual"] - comparison_2025["SARIMA"]) / comparison_2025["Actual"] * 100).round(2)
comparison_2025["Naive_Error"] = (abs(comparison_2025["Actual"] - comparison_2025["Naive_Seasonal"]) / comparison_2025["Actual"] * 100).round(2)
comparison_2025["Ensemble_Error"] = (abs(comparison_2025["Actual"] - comparison_2025["Ensemble"]) / comparison_2025["Actual"] * 100).round(2)

print(comparison_2025[["Month", "Actual", "SARIMA", "Naive_Seasonal", "Ensemble",
                        "SARIMA_Error", "Naive_Error", "Ensemble_Error"]])

print("\n--- Mean errors across all 12 months of 2025 ---")
print("SARIMA:        ", round(comparison_2025["SARIMA_Error"].mean(), 2), "%")
print("Naive Seasonal:", round(comparison_2025["Naive_Error"].mean(), 2), "%")
print("Ensemble:      ", round(comparison_2025["Ensemble_Error"].mean(), 2), "%")

comparison_2025.to_csv("data/full_2025_backtest_comparison.csv", index=False)
print("\nSaved full_2025_backtest_comparison.csv")

      Month  Actual   SARIMA  Naive_Seasonal  Ensemble  SARIMA_Error  \
0   2025-01   73059  77532.0           73751   75263.0          6.12   
1   2025-02   69091  75195.0           73425   74133.0          8.83   
2   2025-03   75599  79458.0           75514   77092.0          5.10   
3   2025-04   74357  76797.0           74024   75133.0          3.28   
4   2025-05   78653  82679.0           78233   80011.0          5.12   
5   2025-06   78414  83438.0           77845   80082.0          6.41   
6   2025-07   82509  84317.0           80099   81786.0          2.19   
7   2025-08   77380  81699.0           78650   79870.0          5.58   
8   2025-09   73857  81107.0           76220   78175.0          9.82   
9   2025-10   76963  82374.0           81476   81835.0          7.03   
10  2025-11   76013  84193.0           80699   82097.0         10.76   
11  2025-12   73393  81159.0           76170   78166.0         10.58   

    Naive_Error  Ensemble_Error  
0          0.95            3.

In [5]:
train_2024 = london_full[london_full["Date"] <= "2023-12-31"]
ts_train_2024 = train_2024.set_index("Date")["CrimeCount"]

print("Training data ends at:", ts_train_2024.index.max())
print("Training data length:", len(ts_train_2024), "months")

model_2024 = SARIMAX(ts_train_2024, order=(1,1,0), seasonal_order=(1,1,0,12),
                      enforce_stationarity=False, enforce_invertibility=False)
fit_2024 = model_2024.fit(disp=False)
forecast_2024 = fit_2024.forecast(steps=12)

actual_2024 = london_full[(london_full["Date"] >= "2024-01-01") & (london_full["Date"] <= "2024-12-31")]
actual_2024 = actual_2024.sort_values("Date").reset_index(drop=True)

naive_2024 = london_full[(london_full["Date"] >= "2023-01-01") & (london_full["Date"] <= "2023-12-31")]
naive_2024 = naive_2024.sort_values("Date").reset_index(drop=True)

print("\nActual 2024 rows:", len(actual_2024))
print("Naive (2023) rows:", len(naive_2024))

Training data ends at: 2023-12-01 00:00:00
Training data length: 165 months

Actual 2024 rows: 12
Naive (2023) rows: 12


/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


In [15]:
comparison_2024 = pd.DataFrame({
    "Month": actual_2024["Date"].dt.strftime("%Y-%m"),
    "Actual": actual_2024["CrimeCount"].values,
    "SARIMA": forecast_2024.round(0).values,
    "Naive_Seasonal": naive_2024["CrimeCount"].values
})

comparison_2024["Ensemble"] = (0.4 * comparison_2024["SARIMA"]) + (0.6 * comparison_2024["Naive_Seasonal"])
comparison_2024["Ensemble"] = comparison_2024["Ensemble"].round(0)

comparison_2024["SARIMA_Error"] = (abs(comparison_2024["Actual"] - comparison_2024["SARIMA"]) / comparison_2024["Actual"] * 100).round(2)
comparison_2024["Naive_Error"] = (abs(comparison_2024["Actual"] - comparison_2024["Naive_Seasonal"]) / comparison_2024["Actual"] * 100).round(2)
comparison_2024["Ensemble_Error"] = (abs(comparison_2024["Actual"] - comparison_2024["Ensemble"]) / comparison_2024["Actual"] * 100).round(2)

print(comparison_2024[["Month", "Actual", "SARIMA", "Naive_Seasonal", "Ensemble",
                        "SARIMA_Error", "Naive_Error", "Ensemble_Error"]])

print("\n--- Mean errors across all 12 months of 2024 ---")
print("SARIMA:        ", round(comparison_2024["SARIMA_Error"].mean(), 2), "%")
print("Naive Seasonal:", round(comparison_2024["Naive_Error"].mean(), 2), "%")
print("Ensemble:      ", round(comparison_2024["Ensemble_Error"].mean(), 2), "%")

comparison_2024.to_csv("data/full_2024_backtest_comparison.csv", index=False)
print("\nSaved full_2024_backtest_comparison.csv")

      Month  Actual   SARIMA  Naive_Seasonal  Ensemble  SARIMA_Error  \
0   2024-01   73751  78675.0           71476   74356.0          6.68   
1   2024-02   73425  75413.0           67504   70668.0          2.71   
2   2024-03   75514  82189.0           73536   76997.0          8.84   
3   2024-04   74024  78029.0           69921   73164.0          5.41   
4   2024-05   78233  85009.0           77165   80303.0          8.66   
5   2024-06   77845  84631.0           78857   81167.0          8.72   
6   2024-07   80099  85514.0           78617   81376.0          6.76   
7   2024-08   78650  82950.0           75049   78209.0          5.47   
8   2024-09   76220  81312.0           75951   78095.0          6.68   
9   2024-10   81476  84107.0           73973   78027.0          3.23   
10  2024-11   80699  85607.0           77904   80985.0          6.08   
11  2024-12   76170  81415.0           76086   78218.0          6.89   

    Naive_Error  Ensemble_Error  
0          3.08            0.

In [9]:
train_2023 = london_full[london_full["Date"] <= "2022-12-31"]
ts_train_2023 = train_2023.set_index("Date")["CrimeCount"]

print("Training data ends at:", ts_train_2023.index.max())
print("Training data length:", len(ts_train_2023), "months")

model_2023 = SARIMAX(ts_train_2023, order=(1,1,0), seasonal_order=(1,1,0,12),
                      enforce_stationarity=False, enforce_invertibility=False)
fit_2023 = model_2023.fit(disp=False)
forecast_2023 = fit_2023.forecast(steps=12)

actual_2023 = london_full[(london_full["Date"] >= "2023-01-01") & (london_full["Date"] <= "2023-12-31")]
actual_2023 = actual_2023.sort_values("Date").reset_index(drop=True)

naive_2023 = london_full[(london_full["Date"] >= "2022-01-01") & (london_full["Date"] <= "2022-12-31")]
naive_2023 = naive_2023.sort_values("Date").reset_index(drop=True)

print("\nActual 2023 rows:", len(actual_2023))
print("Naive (2022) rows:", len(naive_2023))

Training data ends at: 2022-12-01 00:00:00
Training data length: 153 months

Actual 2023 rows: 12
Naive (2022) rows: 12


/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


In [16]:
comparison_2023 = pd.DataFrame({
    "Month": actual_2023["Date"].dt.strftime("%Y-%m"),
    "Actual": actual_2023["CrimeCount"].values,
    "SARIMA": forecast_2023.round(0).values,
    "Naive_Seasonal": naive_2023["CrimeCount"].values
})

comparison_2023["Ensemble"] = (0.4 * comparison_2023["SARIMA"]) + (0.6 * comparison_2023["Naive_Seasonal"])
comparison_2023["Ensemble"] = comparison_2023["Ensemble"].round(0)

comparison_2023["SARIMA_Error"] = (abs(comparison_2023["Actual"] - comparison_2023["SARIMA"]) / comparison_2023["Actual"] * 100).round(2)
comparison_2023["Naive_Error"] = (abs(comparison_2023["Actual"] - comparison_2023["Naive_Seasonal"]) / comparison_2023["Actual"] * 100).round(2)
comparison_2023["Ensemble_Error"] = (abs(comparison_2023["Actual"] - comparison_2023["Ensemble"]) / comparison_2023["Actual"] * 100).round(2)

print(comparison_2023[["Month", "Actual", "SARIMA", "Naive_Seasonal", "Ensemble",
                        "SARIMA_Error", "Naive_Error", "Ensemble_Error"]])

print("\n--- Mean errors across all 12 months of 2023 ---")
print("SARIMA:        ", round(comparison_2023["SARIMA_Error"].mean(), 2), "%")
print("Naive Seasonal:", round(comparison_2023["Naive_Error"].mean(), 2), "%")
print("Ensemble:      ", round(comparison_2023["Ensemble_Error"].mean(), 2), "%")

comparison_2023.to_csv("data/full_2023_backtest_comparison.csv", index=False)
print("\nSaved full_2023_backtest_comparison.csv")

      Month  Actual   SARIMA  Naive_Seasonal  Ensemble  SARIMA_Error  \
0   2023-01   71476  63675.0           66028   65087.0         10.91   
1   2023-02   67504  62276.0           63380   62938.0          7.74   
2   2023-03   73536  70958.0           70805   70866.0          3.51   
3   2023-04   69921  68458.0           66170   67085.0          2.09   
4   2023-05   77165  74611.0           72920   73596.0          3.31   
5   2023-06   78857  74853.0           70736   72383.0          5.08   
6   2023-07   78617  76351.0           72600   74100.0          2.88   
7   2023-08   75049  73242.0           70910   71843.0          2.41   
8   2023-09   75951  72920.0           67058   69403.0          3.99   
9   2023-10   73973  79073.0           74016   76039.0          6.89   
10  2023-11   77904  77812.0           73395   75162.0          0.12   
11  2023-12   76086  71354.0           67132   68821.0          6.22   

    Naive_Error  Ensemble_Error  
0          7.62            8.